# Human in the Loop Agent (my first one!)

In [1]:
!uv pip install -U langchain langchain_ollama langgraph-checkpoint-sqlite

Using Python 3.12.3 environment at: /home/acm/envs/vllmvenv
Resolved 44 packages in 136ms                                        
Audited 44 packages in 0.22ms


In [2]:
from langgraph.graph import StateGraph, END 
from typing import TypedDict, Annotated
import operator 
from uuid import uuid4

from langchain_core.messages import AnyMessage, ToolMessage, SystemMessage, HumanMessage 
from langchain_core.tools import tool

from langchain_ollama import ChatOllama
llm = "qwen3.8:27b"

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)



In [3]:
# tools 
@tool
def get_weather(city: str) -> str:
    """Get current weather conditions for a single city."""
    import requests
    resp = requests.get(f"https://wttr.in/{city}?format=3")
    return resp.text
    
@tool
def get_search_query(query: str) -> str: 
    """Searches web for a query and returns search results."""
    try: 
        results = DDGS().text(query, max_results=15)
    except Exception as e:
        return f"Error calling tool: {e}"
    if not results:
        return "No results found" 
    return "\n\n".join(
        f"Title: {r['title']}\nURl: {r['href']}\n Body: {r['body']}"
        for r in results
    )

In [4]:
"""
Instead of appending new messages, if a new message comes in with teh same id, replace it. 
"""

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign any messages an id if it doesnt exist already 
    for message in right: 
        if not message.id:
            message.id = str(uuid4())

    merged = left.copy()
    for message in right: 
        for i, existing in enumerate(left):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged

class AgentState(TypedDict): 
    messages: Annotated[list[AnyMessage], reduce_messages]

In [5]:
# major change -> before each action node is called, we interrupt. 

class Agent: 
    def __init__(self, model, tools, checkpointer, system=""): 
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_ollama)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True:"action", False: END})
        graph.add_edge("action", "llm") 
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"])
        self.tools = {t.name: t for t in tools} 
        self.model = model.bind_tools(tools) 
        
    def call_ollama(self, state:AgentState): 
        messages = state['messages'] 
        if self.system: 
            messages = [SystemMessage(content=self.system)] + messages 
        message = self.model.invoke(messages) 
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [6]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOllama(model=llm)  
abot = Agent(model, [get_weather, get_search_query], system=prompt, checkpointer=memory)

In [7]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T06:10:35.069012532Z', 'done': True, 'done_reason': 'stop', 'total_duration': 518304932, 'load_duration': 1409275, 'prompt_eval_count': 402, 'prompt_eval_duration': 226476000, 'eval_count': 59, 'eval_duration': 287159000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e2f-11b5-7443-a81f-a5dc72031fa4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '1f52748c-0d88-4c5c-92ea-6f4f14261204', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 59, 'total_tokens': 461})]}
()


In [8]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='5f1c3084-a767-47ec-a830-c8185c51ec4b'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T06:10:35.069012532Z', 'done': True, 'done_reason': 'stop', 'total_duration': 518304932, 'load_duration': 1409275, 'prompt_eval_count': 402, 'prompt_eval_duration': 226476000, 'eval_count': 59, 'eval_duration': 287159000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e2f-11b5-7443-a81f-a5dc72031fa4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '1f52748c-0d88-4c5c-92ea-6f4f14261204', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 59, 'total_tokens': 461})]}, next=('action',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c85f2-2747-64c6-

In [9]:
abot.graph.get_state(thread).next

('action',)

In [10]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '1f52748c-0d88-4c5c-92ea-6f4f14261204', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='San Francisco: ☀️  +61°F\n', name='get_weather', id='26593a59-59c3-4fc0-b5f0-e31c265d8205', tool_call_id='1f52748c-0d88-4c5c-92ea-6f4f14261204')]}
{'messages': [AIMessage(content="Here's the current weather in San Francisco: ☀️ **Sunny, 61°F**\n\nA pleasant, clear day! Let me know if you need anything else.", additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T06:13:29.527097022Z', 'done': True, 'done_reason': 'stop', 'total_duration': 559955984, 'load_duration': 1511883, 'prompt_eval_count': 457, 'prompt_eval_duration': 154428000, 'eval_count': 62, 'eval_duration': 401082000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e31-bb06-7573-ad3e-50f383a728e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'inpu

In [11]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='5f1c3084-a767-47ec-a830-c8185c51ec4b'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T06:10:35.069012532Z', 'done': True, 'done_reason': 'stop', 'total_duration': 518304932, 'load_duration': 1409275, 'prompt_eval_count': 402, 'prompt_eval_duration': 226476000, 'eval_count': 59, 'eval_duration': 287159000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e2f-11b5-7443-a81f-a5dc72031fa4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '1f52748c-0d88-4c5c-92ea-6f4f14261204', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 59, 'total_tokens': 461}), ToolMessage(content='San Francisco: ☀️  +61°F\n', name='get_weather', id='26593a59-59c3-4fc0-b5f0-e31c265d8205', tool_cal

In [12]:
abot.graph.get_state(thread).next

()

In [14]:
messages = [HumanMessage("What's the weather in London?")] 
thread = {"configurable": {"thread_id":"2"}}
for event in abot.graph.stream({"messages": messages}, thread):
     for v in event.values():
         print(v)
while abot.graph.get_state(thread).next:
    print(abot.graph.get_state(thread))
    _input = input("proceed? y/n: ")
    if _input != "y":
        print("aborting...")
        break
    for event in abot.graph.stream(None, thread):
     for v in event.values():
         print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T06:21:47.572425699Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4130294370, 'load_duration': 3478297614, 'prompt_eval_count': 403, 'prompt_eval_duration': 215416000, 'eval_count': 65, 'eval_duration': 433872000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e39-4691-7a23-a4bb-aba94f21cccf-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'id': '5b18a3a5-d786-456e-9977-669a4900560d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 403, 'output_tokens': 65, 'total_tokens': 468})]}
()
StateSnapshot(values={'messages': [HumanMessage(content="What's the weather in London?", additional_kwargs={}, response_metadata={}, id='93aa480d-1582-4bcf-b79d-bf8d04a774ab'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': 

proceed? y/n:  y


Calling: {'name': 'get_weather', 'args': {'city': 'London'}, 'id': '5b18a3a5-d786-456e-9977-669a4900560d', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='London: ☁️  +60°F\n', name='get_weather', id='c20b4576-022d-4a59-abc5-976657caa13b', tool_call_id='5b18a3a5-d786-456e-9977-669a4900560d')]}
{'messages': [AIMessage(content='London is currently cloudy with a temperature of **60°F** (about 15.6°C).', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-10-15T06:21:52.274373008Z', 'done': True, 'done_reason': 'stop', 'total_duration': 438473350, 'load_duration': 1380813, 'prompt_eval_count': 456, 'prompt_eval_duration': 137530000, 'eval_count': 40, 'eval_duration': 295500000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a13e39-675b-7da2-97b7-a1b0b7f4fcdf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 456, 'output_tokens': 40, 'total_tokens': 496})]}
